# 3.3 — Interannual variability assessment — Evapotranspiration

The analysis assesses annual actual evapotranspiration variability independently for every configured dataset. For $n$ valid annual values $X_t$, the analysis calculates the multiannual mean and sample interannual variance:

$$
\overline{X}=\frac{1}{n}\sum_tX_t,\qquad
s^2=\frac{1}{n-1}\sum_t(X_t-\overline{X})^2.
$$

The delete-one jackknife variance of the mean reduces algebraically to the sample variance divided by the valid sample count:

$$
\widehat{\mathrm{Var}}_{JK}(\overline{X})=\frac{s^2}{n}.
$$

The analysis also calculates the standard error $SE=\sqrt{s^2/n}$ and relative uncertainty $RU=100\,SE/\overline{X}$. Variance outputs have units of mm², standard error has units of mm, and relative uncertainty is a percentage.


## Annual stacks, target grid, and common mask

The workflow loads the inclusive annual period from `config.yml`, uses the configured reference dataset as the target grid, and aligns every other annual raster to that grid. It then applies one common finite and positive cell-year mask to all datasets. Consequently, all dataset estimates at a given cell are based on the same valid years.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import xarray as xr

REPOSITORY_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from interannual_variability import common_valid_stacks, variability_metrics, write_variability_set
from accuracy_assessment import enabled_items, load_config, load_stack, open_raster, raster_path, years_for
from regional_plots import plot_grouped_violins

COMPONENT = "evapotranspiration"
config = load_config(REPOSITORY_ROOT / "config.yml")
variability_config = config["interannual_variability"][COMPONENT]
datasets = enabled_items(variability_config, "datasets")
years = years_for(
    config,
    "annual",
    COMPONENT,
    assessment="interannual_variability",
)
reference_key = variability_config["grid_reference"]
reference_dataset = datasets[reference_key]
reference_layers = [
    open_raster(raster_path(reference_dataset, "annual", year)) for year in years
]
stacks = {
    reference_key: xr.concat(
        [layer.expand_dims(year=[year]) for layer, year in zip(reference_layers, years)],
        dim="year",
    )
}
for dataset_key, dataset in datasets.items():
    if dataset_key == reference_key:
        continue
    stacks[dataset_key] = load_stack(
        dataset,
        "annual",
        years,
        match=reference_layers,
        resampling=config["processing"].get("resampling", "bilinear"),
    )
stacks = common_valid_stacks(
    stacks,
    positive_only=config["processing"].get("require_positive_values", True),
)
print(f"{COMPONENT} variability period: {years[0]}–{years[-1]}")


## Variability raster sets

The workflow writes seven rasters for each dataset: valid-year count, multiannual mean, interannual variance, jackknife variance of the mean, standard error of the mean, relative-uncertainty fraction, and relative-uncertainty percent. Nodata remains nodata when fewer than two valid annual values are available for variance estimation.


In [ ]:
output_dir = Path(config["output_dir"]) / "interannual_variability" / COMPONENT
variability_paths = {}
for dataset_key, stack in stacks.items():
    metrics = variability_metrics(stack, dim="year")
    variability_paths[dataset_key] = write_variability_set(
        metrics,
        output_dir / dataset_key,
    )
    print("Completed", dataset_key)


## Climate-region multiannual mean actual evapotranspiration

The workflow extracts the multiannual mean actual evapotranspiration raster cells by climate region for every dataset. The grouped violins describe the spatial distribution of long-term mean ET, not the temporal distribution of annual observations.


In [ ]:
regions = gpd.read_file(config["regions"]["file"])
region_config = config["regions"]
figure_config = config["figures"]
component_figure_config = figure_config["interannual_variability"][COMPONENT]
colors = figure_config["colors"][:len(datasets)]
mean_series = [
    (datasets[key]["label"], variability_paths[key]["multiannual_mean"])
    for key in datasets
]
mean_stats = plot_grouped_violins(
    mean_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / "multiannual_mean_violin.png",
    output_dir / "tables" / "multiannual_mean_regional_statistics.csv",
    "Multiannual mean (mm)",
    tuple(component_figure_config["mean_range"]),
    colors,
    dpi=figure_config["dpi"],
    exclude_zero=False,
)
mean_stats


## Climate-region standard error of the mean

The analysis summarizes standard error, defined as the square root of the jackknife variance of the mean, by climate region. This quantity measures uncertainty in the estimated multiannual mean and is distinct from the interannual standard deviation of individual years.


In [ ]:
se_series = [
    (datasets[key]["label"], variability_paths[key]["standard_error_of_mean"])
    for key in datasets
]
se_stats = plot_grouped_violins(
    se_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / "standard_error_violin.png",
    output_dir / "tables" / "standard_error_regional_statistics.csv",
    "Standard error of mean (mm)",
    tuple(component_figure_config["standard_error_range"]),
    colors,
    dpi=figure_config["dpi"],
    exclude_zero=False,
)
se_stats


## Climate-region relative uncertainty

The analysis divides the standard error by the corresponding multiannual mean and expressed the result as a percentage. This standardized uncertainty allowed datasets with different mean water-depth magnitudes to be compared on the same scale.


In [ ]:
ru_series = [
    (datasets[key]["label"], variability_paths[key]["relative_uncertainty_percent"])
    for key in datasets
]
ru_stats = plot_grouped_violins(
    ru_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / "relative_uncertainty_violin.png",
    output_dir / "tables" / "relative_uncertainty_regional_statistics.csv",
    "Relative uncertainty (%)",
    tuple(component_figure_config["relative_uncertainty_range"]),
    colors,
    dpi=figure_config["dpi"],
    exclude_zero=False,
)
ru_stats
